# Generative AI - Assignment 1
Name : Yashraj Singh Srinet,
27PGAI0014


In [1]:
!pip install -q -U langchain langchain-core langchain-community langchain-ollama pandas


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_ollama import ChatOllama
import pandas as pd
import time
import json
import warnings
warnings.filterwarnings("ignore")

llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
)

In [3]:
import time


def safe_run(chain, **kwargs):
    """Run a chain and return the text content."""
    result = chain.invoke(kwargs)
    return result.content

def truncate_text(text, max_chars=1500):
    """Truncate text (kept for consistency; not strictly needed with local Ollama)."""
    return str(text)[:max_chars]


In [4]:

bbc_df = pd.read_csv("bbc-news-data.csv", sep="\t")
print(f"Full dataset shape: {bbc_df.shape}")
print(f"Columns: {list(bbc_df.columns)}")
print(f"Categories: {bbc_df['category'].unique()}")
bbc_df.head()


Full dataset shape: (2225, 4)
Columns: ['category', 'filename', 'title', 'content']
Categories: ['business' 'entertainment' 'politics' 'sport' 'tech']


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [5]:
# Limit to first 30 articles as instructed
df_news = bbc_df.head(30).copy().reset_index(drop=True)
print(f"Working dataset shape: {df_news.shape}")
df_news.head()


Working dataset shape: (30, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [6]:
from langchain_core.prompts import PromptTemplate


In [7]:

classification_prompt = PromptTemplate(
    input_variables=["article"],
    template="""Analyze the following news article and identify its topic as exactly one of the following categories:
Business, Entertainment, Politics, Sport, Tech.

Few-shot examples:
Article: "Quarterly profits at US media giant TimeWarner jumped 76%..."
Topic: Business

Article: "Robbie Williams rocks the MTV music awards ceremony..."
Topic: Entertainment

Now classify:
Article: \"{article}\"

Respond with ONLY the single category label (Business, Entertainment, Politics, Sport, or Tech). Nothing else."""
)

classification_chain = classification_prompt | llm


In [8]:
sample_article = df_news.loc[0, "content"]
sample_result = classification_chain.invoke({"article": sample_article}).content
print(f"Article title : {df_news.loc[0, 'title']}")
print(f"Actual category: {df_news.loc[0, 'category']}")
print(f"Detected topic : {sample_result.strip()}")

Article title : Ad sales boost Time Warner profit
Actual category: business
Detected topic : Business


## Summarization



In [9]:
# Define the summarization prompt
summary_prompt = PromptTemplate(
    input_variables=["article"],
    template="""Summarize the main points of the following news article in 2-3 sentences.
Capture the key facts (who, what, when, where, why) without personal commentary.

Article:
\"{article}\"

Summary:"""
)

summary_chain = summary_prompt | llm


In [10]:
# Show it works for a sample datapoint
sample_summary = safe_run(summary_chain, article=truncate_text(sample_article))
print(f"Article title: {df_news.loc[0, 'title']}")
print(f"Summary      : {sample_summary.strip()}")


Article title: Ad sales boost Time Warner profit
Summary      : Warner Bros. reported a 2% increase in fourth-quarter profits to $11.1 billion, driven by higher internet advertising revenue and a one-time gain. Despite a 2% decrease in overall profits, the company’s internet business, particularly AOL, showed mixed results with subscriber losses and stronger advertising.  The firm’s film division experienced a significant decline, impacting overall profitability.


##  Key Entity Extraction 



In [11]:
# Define the entity extraction prompt
entity_prompt = PromptTemplate(
    input_variables=["article"],
    template="""From the following news article, list the names of any important people, organizations, or places mentioned.
Return ONLY a JSON list of strings. Example: ["Google", "London", "John Smith"]

Article:
\"{article}\"

Entities (JSON list):"""
)

entity_chain = entity_prompt | llm


In [12]:
# Show it works for a sample datapoint
sample_entities = safe_run(entity_chain, article=truncate_text(sample_article))
print(f"Article title: {df_news.loc[0, 'title']}")
print(f"Key Entities : {sample_entities.strip()}")


Article title: Ad sales boost Time Warner profit
Key Entities : ```json
[
  "TimeWarner",
  "US media giant",
  "Google",
  "Warner Bros",
  "AOL",
  "US Securities Exchange Commission (SEC)",
  "Alexander",
  "Catwoman"
]
```


##  Update the DataFrame with Results



In [13]:
def parse_entities(raw_str):
    """Try to parse entity string as JSON list; fall back to raw string."""
    try:
        entities = json.loads(raw_str.strip())
        if isinstance(entities, list):
            return entities
    except json.JSONDecodeError:
        pass
    return raw_str.strip()

# Lists to collect results
detected_topics = []
summaries = []
key_entities = []

for idx, row in df_news.iterrows():
    article_text = truncate_text(row["content"])
    print(f"Processing article {idx + 1}/30: {row['title'][:50]}...")

    # Classification
    topic = safe_run(classification_chain, article=article_text).strip()
    detected_topics.append(topic)
    time.sleep(3)

    # Summarization
    summary = safe_run(summary_chain, article=article_text).strip()
    summaries.append(summary)
    time.sleep(3)

    # Entity extraction
    entities_raw = safe_run(entity_chain, article=article_text).strip()
    entities = parse_entities(entities_raw)
    key_entities.append(entities)
    time.sleep(3)

print("\nDone processing all 30 articles.")

Processing article 1/30: Ad sales boost Time Warner profit...
Processing article 2/30: Dollar gains on Greenspan speech...
Processing article 3/30: Yukos unit buyer faces loan claim...
Processing article 4/30: High fuel prices hit BA's profits...
Processing article 5/30: Pernod takeover talk lifts Domecq...
Processing article 6/30: Japan narrowly escapes recession...
Processing article 7/30: Jobs growth still slow in the US...
Processing article 8/30: India calls for fair trade rules...
Processing article 9/30: Ethiopia's crop production up 24%...
Processing article 10/30: Court rejects $280bn tobacco case...
Processing article 11/30: Ask Jeeves tips online ad revival...
Processing article 12/30: Indonesians face fuel price rise...
Processing article 13/30: Peugeot deal boosts Mitsubishi...
Processing article 14/30: Telegraph newspapers axe 90 jobs...
Processing article 15/30: Air passengers win new EU rights...
Processing article 16/30: China keeps tight rein on credit...
Processing a

In [14]:
# Add new columns to the DataFrame
df_news["Detected_Topic"] = detected_topics
df_news["Summary"] = summaries
df_news["Key_Entities"] = key_entities

print(f"DataFrame shape: {df_news.shape}")
print(f"Columns: {list(df_news.columns)}")
df_news[["title", "category", "Detected_Topic", "Summary", "Key_Entities"]].head(10)


DataFrame shape: (30, 7)
Columns: ['category', 'filename', 'title', 'content', 'Detected_Topic', 'Summary', 'Key_Entities']


,title,category,Detected_Topic,Summary,Key_Entities
0,Ad sales boost Time Warner profit,business,Business,Warner Bros. reported a 76% increase in quarte...,"```json\n[\n ""TimeWarner"",\n ""US media giant..."
1,Dollar gains on Greenspan speech,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""Federal Reserve"",\n ""Alan Gree..."
2,Yukos unit buyer faces loan claim,business,Business,"In December, the owners of Yukos are seeking r...","```json\n[\n ""Yukos"",\n ""Menatep Group"",\n ..."
3,High fuel prices hit BA's profits,business,Business,British Airways experienced a 40% drop in prof...,"```json\n[\n ""Rod Eddington"",\n ""British Air..."
4,Pernod takeover talk lifts Domecq,business,Business,Here’s a 2-3 sentence summary of the news arti...,"```json\n[\n ""Allied Domecq"",\n ""Pernod Rica..."
5,Japan narrowly escapes recession,business,Business,Japan’s economy experienced a minor slowdown i...,"```json\n[\n ""Japan"",\n ""Heizo Takenaka"",\n ..."
6,Jobs growth still slow in the US,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""US"",\n ""Labor Department"",\n ..."
7,India calls for fair trade rules,business,Business,"India is attending the G7 meeting in London, d...","```json\n[\n ""India"",\n ""London"",\n ""G7"",\n..."
8,Ethiopia's crop production up 24%,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""Ethiopia"",\n ""Food and Agricul..."
9,Court rejects $280bn tobacco case,business,Business,A US government appeal court rejected a lawsui...,"```json\n[\n ""Altria Group"",\n ""RJ Reynolds ..."


In [15]:
# Display the final merged DataFrame with all original and new columns
df_news.head()


,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,Warner Bros. reported a 76% increase in quarte...,"```json\n[\n ""TimeWarner"",\n ""US media giant..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""Federal Reserve"",\n ""Alan Gree..."
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,"In December, the owners of Yukos are seeking r...","```json\n[\n ""Yukos"",\n ""Menatep Group"",\n ..."
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways experienced a 40% drop in prof...,"```json\n[\n ""Rod Eddington"",\n ""British Air..."
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Here’s a 2-3 sentence summary of the news arti...,"```json\n[\n ""Allied Domecq"",\n ""Pernod Rica..."


---
# Part 2: Job Postings Analysis – Role Categorization & Requirements Extraction


In [16]:
# Load job postings dataset
jobs_df = pd.read_csv("job_title_des.csv")
print(f"Full dataset shape: {jobs_df.shape}")
print(f"Columns: {list(jobs_df.columns)}")
jobs_df.head()


Full dataset shape: (2277, 3)
Columns: ['Unnamed: 0', 'Job Title', 'Job Description']


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [17]:
# Rename columns for consistency and limit to first 25
jobs_df = jobs_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
# Drop the unnamed index column if present
if "" in jobs_df.columns or "Unnamed: 0" in jobs_df.columns:
    jobs_df = jobs_df.drop(columns=[c for c in jobs_df.columns if "Unnamed" in c or c == ""], errors="ignore")

df_jobs = jobs_df.head(25).copy().reset_index(drop=True)
print(f"Working dataset shape: {df_jobs.shape}")
df_jobs.head()


Working dataset shape: (25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2 – Job Category Classification



In [18]:
# Define the job category classification prompt
job_category_prompt = PromptTemplate(
    input_variables=["job_title", "job_description"],
    template="""Given the following job title and description, categorize the job into exactly one of these domains:
Technology/IT, Finance, Marketing, Healthcare, Education, Others.

Few-shot examples:
Job Title: Software Engineer
Description: Build scalable web applications using React and Node.js...
Category: Technology/IT

Job Title: Registered Nurse
Description: Provide patient care in a clinical setting...
Category: Healthcare

Now classify:
Job Title: {job_title}
Description: {job_description}

Respond with ONLY the single category label. Nothing else."""
)

job_category_chain = job_category_prompt | llm


In [19]:
# Show it works for a sample datapoint
sample_job = df_jobs.loc[0]
sample_cat = safe_run(
    job_category_chain,
    job_title=sample_job["Job_Title"],
    job_description=truncate_text(sample_job["Job_Description"])
).strip()
print(f"Job Title         : {sample_job['Job_Title']}")
print(f"Predicted Category: {sample_cat}")


Job Title         : Flutter Developer
Predicted Category: Technology/IT


##  Requirements Extraction 



In [20]:
# Define the requirements extraction prompt (composite)
requirements_prompt = PromptTemplate(
    input_variables=["job_description"],
    template="""Extract the following from the job description below. Respond ONLY in valid JSON with these exact keys:
- "Required_Skills": a list of key skills or technologies mentioned (e.g. ["Python", "SQL", "project management"])
- "Education_Required": the minimum education level required or preferred (e.g. "Bachelor's degree"). If not stated, use "Not specified".
- "Experience_Required": the minimum years of experience or experience level required (e.g. "3+ years"). If not mentioned, use "Not specified".

Job Description:
\"{job_description}\"

JSON output:"""
)

requirements_chain = requirements_prompt | llm


In [21]:
# Show it works for a sample datapoint
sample_req_raw = safe_run(
    requirements_chain,
    job_description=truncate_text(sample_job["Job_Description"])
).strip()
print(f"Job Title: {sample_job['Job_Title']}")
print(f"Extracted Requirements:\n{sample_req_raw}")


Job Title: Flutter Developer
Extracted Requirements:
```json
{
  "Required_Skills": [
    "Flutter",
    "JavaScript",
    "Dart"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "1 year"
}
```


In [22]:
# Parse the JSON output
def parse_requirements(raw_str):
    """Parse the LLM output into a dict with required fields."""
    defaults = {
        "Required_Skills": [],
        "Education_Required": "Not specified",
        "Experience_Required": "Not specified"
    }
    try:
        # Remove markdown code fences if present
        cleaned = raw_str.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
        parsed = json.loads(cleaned)
        for key in defaults:
            if key not in parsed or parsed[key] is None:
                parsed[key] = defaults[key]
        return parsed
    except (json.JSONDecodeError, Exception):
        return defaults

# Demonstrate parsing
parsed_sample = parse_requirements(sample_req_raw)
print(json.dumps(parsed_sample, indent=2))


{
  "Required_Skills": [
    "Flutter",
    "JavaScript",
    "Dart"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "1 year"
}


In [23]:
predicted_categories = []
required_skills_list = []
education_required_list = []
experience_required_list = []

for idx, row in df_jobs.iterrows():
    title = row["Job_Title"]
    desc = truncate_text(row["Job_Description"])
    print(f"Processing job {idx + 1}/25: {title[:50]}...")

    cat = safe_run(job_category_chain, job_title=title, job_description=desc).strip()
    predicted_categories.append(cat)
    time.sleep(3)

    req_raw = safe_run(requirements_chain, job_description=desc).strip()
    req = parse_requirements(req_raw)
    required_skills_list.append(req["Required_Skills"])
    education_required_list.append(req["Education_Required"])
    experience_required_list.append(req["Experience_Required"])
    time.sleep(3)

print("\nDone processing all 25 job postings.")

Processing job 1/25: Flutter Developer...
Processing job 2/25: Django Developer...
Processing job 3/25: Machine Learning...
Processing job 4/25: iOS Developer...
Processing job 5/25: Full Stack Developer...
Processing job 6/25: Java Developer...
Processing job 7/25: Full Stack Developer...
Processing job 8/25: JavaScript Developer...
Processing job 9/25: DevOps Engineer...
Processing job 10/25: Software Engineer...
Processing job 11/25: Database Administrator...
Processing job 12/25: Machine Learning...
Processing job 13/25: Machine Learning...
Processing job 14/25: Software Engineer...
Processing job 15/25: Software Engineer...
Processing job 16/25: Java Developer...
Processing job 17/25: Wordpress Developer...
Processing job 18/25: iOS Developer...
Processing job 19/25: Database Administrator...
Processing job 20/25: DevOps Engineer...
Processing job 21/25: Database Administrator...
Processing job 22/25: Wordpress Developer...
Processing job 23/25: JavaScript Developer...
Processing 

## Update the DataFrame with New Columns



In [24]:
# Add new columns
df_jobs["Predicted_Category"] = predicted_categories
df_jobs["Required_Skills"] = required_skills_list
df_jobs["Education_Required"] = education_required_list
df_jobs["Experience_Required"] = experience_required_list

print(f"DataFrame shape: {df_jobs.shape}")
print(f"Columns: {list(df_jobs.columns)}")
df_jobs[["Job_Title", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]].head(10)


DataFrame shape: (25, 6)
Columns: ['Job_Title', 'Job_Description', 'Predicted_Category', 'Required_Skills', 'Education_Required', 'Experience_Required']


,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,"[Flutter, JavaScript, Dart]",Not specified,1 year
1,Django Developer,Technology/IT,"[Python, Django, REST/RPC, API Frameworks (Dja...",Not specified,Not specified
2,Machine Learning,Technology/IT,"[Python, Machine Learning, Deep Learning, Stat...",Not specified,3+ years
3,iOS Developer,Technology/IT,"[iOS, Objective-C, Cocoa Touch, Core Data, Cor...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"[React, JavaScript, React Native, Redux, Angul...",Not specified,5+ years
5,Java Developer,Technology/IT,"[C#, NET, SQL, Web Services (WSDL, Soap, Restf...",Not specified,Not specified
6,Full Stack Developer,Technology/IT,"[Node.js, Java, JavaScript, MongoDB, Elasticse...",B.sc degree Computer Science Engineering,2 years
7,JavaScript Developer,Technology/IT,"[ReactJS, NodeJs, Azure Functions, GraphQL, Ja...",Not specified,3 - 8 years
8,DevOps Engineer,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,Software Engineer,Technology/IT,"[Software Engineering, REST API, Software Deve...",Not specified,Not specified


In [25]:
# Display the final merged DataFrame with all original and new columns
df_jobs.head()


,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,"[Flutter, JavaScript, Dart]",Not specified,1 year
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, REST/RPC, API Frameworks (Dja...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n...",Technology/IT,"[Python, Machine Learning, Deep Learning, Stat...",Not specified,3+ years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...,Technology/IT,"[iOS, Objective-C, Cocoa Touch, Core Data, Cor...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, JavaScript, React Native, Redux, Angul...",Not specified,5+ years


In [ ]:
from langchain_ollama import ChatOllama

# Initialise Ollama LLM (runs locally – no rate limits)
ollama_llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
)

# Re-create chains with Ollama LLM
classification_chain_ollama = classification_prompt | ollama_llm
summary_chain_ollama = summary_prompt | ollama_llm
entity_chain_ollama = entity_prompt | ollama_llm
job_category_chain_ollama = job_category_prompt | ollama_llm
requirements_chain_ollama = requirements_prompt | ollama_llm


###  ALL BBC News Articles


In [28]:
# Process ALL BBC News articles using Ollama
df_news_all = bbc_df.copy().reset_index(drop=True)
total = len(df_news_all)
print(f"Processing all {total} BBC News articles with Ollama...")

all_topics = []
all_summaries = []
all_entities = []

for idx, row in df_news_all.iterrows():
    article_text = truncate_text(row["content"])
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"  Processing {idx + 1}/{total}...")

    # Classification
    topic = classification_chain_ollama.invoke({"article": article_text}).content.strip()
    all_topics.append(topic)

    # Summarization
    summary = summary_chain_ollama.invoke({"article": article_text}).content.strip()
    all_summaries.append(summary)

    # Entity extraction
    entities_raw = entity_chain_ollama.invoke({"article": article_text}).content.strip()
    entities = parse_entities(entities_raw)
    all_entities.append(entities)

df_news_all["Detected_Topic"] = all_topics
df_news_all["Summary"] = all_summaries
df_news_all["Key_Entities"] = all_entities

print(f"\nDone! Final DataFrame shape: {df_news_all.shape}")
df_news_all[["title", "category", "Detected_Topic", "Summary", "Key_Entities"]].head(10)


Processing all 2225 BBC News articles with Ollama...
  Processing 1/2225...
  Processing 50/2225...
  Processing 100/2225...
  Processing 150/2225...
  Processing 200/2225...
  Processing 250/2225...
  Processing 300/2225...
  Processing 350/2225...
  Processing 400/2225...
  Processing 450/2225...
  Processing 500/2225...
  Processing 550/2225...
  Processing 600/2225...
  Processing 650/2225...
  Processing 700/2225...
  Processing 750/2225...
  Processing 800/2225...
  Processing 850/2225...
  Processing 900/2225...
  Processing 950/2225...
  Processing 1000/2225...
  Processing 1050/2225...
  Processing 1100/2225...
  Processing 1150/2225...
  Processing 1200/2225...
  Processing 1250/2225...
  Processing 1300/2225...
  Processing 1350/2225...
  Processing 1400/2225...
  Processing 1450/2225...
  Processing 1500/2225...
  Processing 1550/2225...
  Processing 1600/2225...
  Processing 1650/2225...
  Processing 1700/2225...
  Processing 1750/2225...
  Processing 1800/2225...
  Proces

,title,category,Detected_Topic,Summary,Key_Entities
0,Ad sales boost Time Warner profit,business,Business,Warner Bros. reported a 76% increase in quarte...,"```json\n[\n ""TimeWarner"",\n ""US media giant..."
1,Dollar gains on Greenspan speech,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""Federal Reserve"",\n ""Alan Gree..."
2,Yukos unit buyer faces loan claim,business,Business,"In December, the owners of Yukos are seeking r...","```json\n[\n ""Yukos"",\n ""Menatep Group"",\n ..."
3,High fuel prices hit BA's profits,business,Business,British Airways experienced a 40% drop in prof...,"```json\n[\n ""Rod Eddington"",\n ""British Air..."
4,Pernod takeover talk lifts Domecq,business,Business,Here’s a 2-3 sentence summary of the news arti...,"```json\n[\n ""Allied Domecq"",\n ""Pernod Rica..."
5,Japan narrowly escapes recession,business,Business,Japan’s economy experienced a minor slowdown i...,"```json\n[\n ""Japan"",\n ""Heizo Takenaka"",\n ..."
6,Jobs growth still slow in the US,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""US"",\n ""Labor Department"",\n ..."
7,India calls for fair trade rules,business,Business,"India is attending the G7 meeting in London, d...","```json\n[\n ""India"",\n ""London"",\n ""G7"",\n..."
8,Ethiopia's crop production up 24%,business,Business,Here’s a 2-3 sentence summary of the article:\...,"```json\n[\n ""Ethiopia"",\n ""Food and Agricul..."
9,Court rejects $280bn tobacco case,business,Business,A US government appeal court rejected a lawsui...,"```json\n[\n ""Altria Group"",\n ""RJ Reynolds ..."


## ALL Job Postings


In [ ]:
# Process ALL job postings using Ollama
df_jobs_all = jobs_df.copy().reset_index(drop=True)
total = len(df_jobs_all)
print(f"Processing all {total} job postings with Ollama...")

all_categories = []
all_skills = []
all_education = []
all_experience = []

for idx, row in df_jobs_all.iterrows():
    title = row["Job_Title"]
    desc = truncate_text(row["Job_Description"])
    if (idx + 1) % 100 == 0 or idx == 0:
        print(f"  Processing {idx + 1}/{total}: {title[:40]}...")

    # Category
    cat = job_category_chain_ollama.invoke({"job_title": title, "job_description": desc}).content.strip()
    all_categories.append(cat)

    # Requirements
    req_raw = requirements_chain_ollama.invoke({"job_description": desc}).content.strip()
    req = parse_requirements(req_raw)
    all_skills.append(req["Required_Skills"])
    all_education.append(req["Education_Required"])
    all_experience.append(req["Experience_Required"])

df_jobs_all["Predicted_Category"] = all_categories
df_jobs_all["Required_Skills"] = all_skills
df_jobs_all["Education_Required"] = all_education
df_jobs_all["Experience_Required"] = all_experience

print(f"\nDone! Final DataFrame shape: {df_jobs_all.shape}")
df_jobs_all[["Job_Title", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]].head(10)

